## CardiaSense AI — Dataset exploration

This notebook covers: **counts, classes, audio playback, spectrograms, signal quality**.

Run `./setup.sh` first (creates the venv, installs deps, downloads the dataset if needed).
The dataset root is resolved from `CARDIASENSE_DATA_DIR` or `.env`.

> Educational / research prototype — not a medical device.


In [1]:

import os, sys
from pathlib import Path

ROOT = Path("/home/tnzr/CardiaSense-AI")
sys.path.insert(0, str(ROOT / "ml"))
sys.path.insert(0, str(ROOT))

DATA = os.environ.get("CARDIASENSE_DATA_DIR") or     "/media/tnzr/AuxVolume/datasets/HLS-CMDS Heart and Lung Sounds Dataset " \
    "Recorded from a Clinical Manikin using Digital Stethoscope"


In [2]:

from cardia.data.hls_cmds import build_task_manifest
m = build_task_manifest(DATA, "sound", "binary")
print("sound:", len(m), "files")
print(m["task_label"].value_counts())
for task in ("heart", "lung"):
    mm = build_task_manifest(DATA, task, "binary")
    print(task, len(mm), "samples")
    print(mm["task_label"].value_counts().to_dict())


sound: 535 files
task_label
heart    195
lung     195
mixed    145
Name: count, dtype: int64
heart 195 samples
{'abnormal': 173, 'normal': 22}
lung 195 samples
{'abnormal': 155, 'normal': 40}


### Listen to a few clips (embedded audio)

In [3]:

import IPython.display as d
from cardia.data.transforms import load_wav, peak_normalize

for label in ("heart", "lung", "mixed"):
    row = m[m["task_label"] == label].iloc[0]
    wav, sr = load_wav(row["file_path"])
    display(d.Audio(wav.numpy(), rate=sr))
    print(label, row["sample_id"])


heart F_N_RC


lung M_N_RUA


mixed M0001


### Waveform + log-mel of one clip

In [4]:

import matplotlib.pyplot as plt
from cardia.data.transforms import MelSpec, preprocess
from cardia.wandb_utils import viz

row = m[m["task_label"] == "heart"].iloc[0]
raw, sr = load_wav(row["file_path"]); raw = peak_normalize(raw)
filt = preprocess(row["file_path"], TYPE_CHECK := None) if False else None
import torch
from cardia.config import Config
cfg = Config()
filt = preprocess(row["file_path"], cfg)
mel = MelSpec(cfg.sample_rate, cfg.n_mels, cfg.n_fft, cfg.hop_length)
viz.waveform_fig(raw.numpy(), sr, f"raw {row['sample_id']}")
plt.show()
viz.spec_fig(mel(filt).numpy(), f"log-mel {row['sample_id']}")
plt.show()


/tmp/ipykernel_963086/4111326953.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_963086/4111326953.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Signal quality indicators (sample)

In [5]:

from cardia.data.stats import sqi_stats
df = sqi_stats(m, n=50)
print(df[["sample_id","duration_s","rms","snr_estimate","clip_ratio"]].describe().T)


              count       mean       std        min        25%        50%  \
duration_s     50.0  15.000000  0.000000  15.000000  15.000000  15.000000   
rms            50.0   0.098548  0.045507   0.014796   0.063869   0.089783   
snr_estimate   50.0   5.413187  1.839675   2.368986   4.139883   5.469579   
clip_ratio     50.0   0.000030  0.000019   0.000000   0.000017   0.000033   

                    75%        max  
duration_s    15.000000  15.000000  
rms            0.128241   0.193469  
snr_estimate   6.418427  10.864118  
clip_ratio     0.000033   0.000067  
